In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
train_df.head(10)

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
print("=== Data Types ===")
print(train_df.dtypes)

print("\n=== Missing Values in Train ===")
print(train_df.isnull().sum())

print("\n=== Missing Values in Test ===")
print(test_df.isnull().sum())

print("\n=== Basic Statistics ===")
train_df.describe()

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
print("=== Overall Survival Rate ===")
print(train_df['Survived'].value_counts())
print(f"\n{train_df['Survived'].mean()*100:.1f}% of passengers survived")

fig, axes = plt.subplots(1,3, figsize=(15,5))

#Gender VS Survival
sns.barplot(data=train_df, x='Sex', y='Survived', ax=axes[0])
axes[0].set_title('Survival Rate by Gender')
axes[0].set_ylabel('Survival Rate')

#Class VS Survival
sns.barplot(data=train_df, x='Pclass', y='Survived', ax=axes[1])
axes[1].set_title('Survival Rate by Class')

train_df['Age'].hist(bins=30, ax=axes[2])
axes[2].set_title('Age Distribution')
axes[2].set_xlabel('Age')

plt.tight_layout()
plt.show()

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))

sns.barplot(data=train_df, x='SibSp', y='Survived', ax=axes[0])
axes[0].set_title('Survival Rate by Siblings/Spouses')

sns.barplot(data=train_df, x='Parch', y='Survived', ax=axes[1])
axes[1].set_title('Survival Rate by Parents/Children')

plt.tight_layout()
plt.show()

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
# Work on copies to avoid modifying originals
train = train_df.copy()
test  = test_df.copy()

#Fill missing age
train['Age'].fillna(train['Age'].median(), inplace=True)
test['Age'].fillna(test['Age'].median(),   inplace=True)

#Fill missing embarked
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)
test['Embarked'].fillna(test['Embarked'].mode()[0],   inplace=True)

#drop cabin column, its useless
train.drop(columns=['Cabin'], inplace=True)
test.drop(columns=['Cabin'],  inplace=True)

print("=== Missing Values After Cleaning ===")
print("Train:")
print(train.isnull().sum())
print("\nTest:")
print(test.isnull().sum())

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title']  = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

print("Titles found:")
print(train['Title'].value_counts())

rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
train['Title'] = train['Title'].replace(rare_titles, 'Rare')
test['Title']  = test['Title'].replace(rare_titles, 'Rare')

# Normalize variations
train['Title'] = train['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
test['Title']  = test['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print("\nCleaned titles:")
print(train['Title'].value_counts())

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize']  = test['SibSp']  + test['Parch']  + 1

train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone']  = (test['FamilySize']  == 1).astype(int)

print("Family size distribution:")
print(train['FamilySize'].value_counts().sort_index())

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
train['Sex'] = LabelEncoder().fit_transform(train['Sex'])
test['Sex']  = LabelEncoder().fit_transform(test['Sex'])

train['Embarked'] = LabelEncoder().fit_transform(train['Embarked'])
test['Embarked']  = LabelEncoder().fit_transform(test['Embarked'])

title_encoder = LabelEncoder()
title_encoder.fit(train['Title'])  # fit on train only
train['Title'] = title_encoder.transform(train['Title'])
test['Title']  = title_encoder.transform(test['Title'])

print(train[['Sex', 'Embarked', 'Title']].head())

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'Title', 'FamilySize', 'IsAlone']
X = train[features]
y = train['Survived']

X_test_final = test[features]

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

print(f"Training samples   : {X_train.shape[0]}")
print(f"Validation samples : {X_val.shape[0]}")
print(f"Features used      : {features}")

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_final)

models = {
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Logistic Regression' : LogisticRegression(random_state=42, max_iter=200),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    if name in ['KNN', 'Logistic Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    results[name] = {
        'model'       : model,
        'predictions' : y_pred,
        'accuracy'    : acc
    }
    print(f"{name:25s} → {acc*100:.2f}%")

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
# Detailed evaluation of each model
for name, result in results.items():
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"Accuracy: {result['accuracy']*100:.2f}%")
    print(classification_report(y_val, result['predictions'],
                                target_names=['Died', 'Survived']))

# Confusion matrices for all models
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Died', 'Survived'],
                yticklabels=['Died', 'Survived'])
    ax.set_title(f"{name}\n{result['accuracy']*100:.1f}%")
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
from sklearn.pipeline import Pipeline

models_cv = {
    'KNN': Pipeline([('scaler', StandardScaler()),
                     ('model', KNeighborsClassifier(n_neighbors=5))]),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                     ('model', LogisticRegression(random_state=42, max_iter=200))]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

print("Cross Validation Results (5-fold):\n")
for name, model in models_cv.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f"{name:25s} → {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/

In [ ]:
best_model = results['Random Forest']['model']
kaggle_predictions = best_model.predict(X_test_final)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived'   : kaggle_predictions
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))

#  ______   __  __   ____    ______   _____     _____
# / _____| |  \/  | |  _ \  |  ____| |  __ \   / ____|
#| |__     | \  / | | |_) | | |__    | |__) | | (___
#|  __|    | |\/| | |  _ <  |  __|   |  _  /   \___ \
#| |____   | |  | | | |_) | | |____  | | \ \   ____) |
# \______| |_|  |_| |____/  |______| |_|  \_\ |_____/